# Unified UAE Real Estate Analytics — Dashboard Graph Analysis

Reproduce, independently, the numbers behind each dashboard visualisation, then
draw the visualisation from those numbers.

Every section follows the same order:

> **DATA → FILTER → CALCULATION → INTERMEDIATE DATAFRAME → RESULT VALUES → VISUALISATION**

The dataframe and the printed values always come **before** the chart, so the
numbers can be checked first and the picture second.

This notebook does **not** import the Streamlit application. It re-implements the
methodology with plain pandas, so agreement with the dashboard is evidence rather
than a tautology.

## Scope

| § | Dashboard visualisation |
|---|---|
| 1 | Transactions recorded each year (year-over-year transaction growth) |
| 2 | How prices are moving |
| 3 | Volume against price |
| 4 | Share of recorded transactions associated with each amenity |
| 5 | Rate by building height and property type |
| 6 | Where the price points are |
| 7 | Rate per m² by layout |
| 8 | Unit size — key statistics |
| 9 | Sale price by registration type — summary |
| 10 | How the price distribution has changed |
| 11 | Year-by-year summary |

## Running it

**Google Colab** — run §0, upload the two parquet files when prompted (or mount
Drive and set `UAE_DATA_DIR`), then Run All.

**Locally** — set `UAE_DATA_DIR` to the folder holding the two parquet files, or
run from the repository root where `data/dubai/` already exists. Then Run All.

| File | What it is | Used for |
|---|---|---|
| `transactions.parquet` | **RAW** registry, 1,762,262 rows | Transaction counts and volume |
| `latest_combined_data.parquet` | **CLEANED** residential-unit sales, 818,838 rows | All price and rate statistics |

**The data-source rule in one line:** counts come from RAW because preprocessing
removes rows from the cleaned file; prices come from CLEANED because that is
where the engineered columns live and it is the validated price basis.

## 0. Setup and data loading

In [ ]:
# Google Colab: uncomment if anything is missing.
# !pip -q install pandas pyarrow plotly

import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# ── Where the two datasets live ─────────────────────────────────────────────
DATA_DIR   = Path(os.environ.get("UAE_DATA_DIR", "data/dubai"))
RAW_FILE   = DATA_DIR / "transactions.parquet"           # RAW registry
CLEAN_FILE = DATA_DIR / "latest_combined_data.parquet"   # CLEANED dataset

try:                       # Colab: offer an upload if the files are not present
    import google.colab                                   # noqa: F401
    if not RAW_FILE.exists():
        from google.colab import files
        print("Upload transactions.parquet and latest_combined_data.parquet")
        files.upload()
        DATA_DIR = Path(".")
        RAW_FILE, CLEAN_FILE = DATA_DIR / "transactions.parquet", DATA_DIR / "latest_combined_data.parquet"
except ImportError:
    pass

for f in (RAW_FILE, CLEAN_FILE):
    print(f"{'OK     ' if f.exists() else 'MISSING'} {f}")

### 0.1 Load both datasets

In [ ]:
# ── RAW registry — four columns only, because it is used for COUNTS ─────────
RAW_COUNT_COLUMNS = ["instance_date", "trans_group_en", "property_type_en", "property_usage_en"]

raw = pd.read_parquet(RAW_FILE, columns=RAW_COUNT_COLUMNS)
raw["instance_date"] = pd.to_datetime(raw["instance_date"], errors="coerce")
raw["year"]  = raw["instance_date"].dt.year
raw["month"] = raw["instance_date"].dt.month
raw = raw.dropna(subset=["year"])

# The same population every Dubai chart covers, taken straight from the
# registry with NO cleaning applied.
RESIDENTIAL_UNIT_SALE = (
    (raw["trans_group_en"]    == "Sales") &
    (raw["property_type_en"]  == "Unit")  &
    (raw["property_usage_en"] == "Residential")
)
raw_res = raw[RESIDENTIAL_UNIT_SALE]

# ── CLEANED dataset — every price and rate figure comes from here ───────────
CLEAN_COLUMNS = ["instance_date", "year", "month", "year_month", "actual_worth",
                 "meter_sale_price", "procedure_area", "rooms_en", "reg_type_en",
                 "building_name_en", "floors",
                 "has_parking", "swimming_pool", "balcony", "elevator", "metro"]
clean = pd.read_parquet(CLEAN_FILE, columns=CLEAN_COLUMNS)
clean["instance_date"] = pd.to_datetime(clean["instance_date"])

print(f"RAW registry          : {len(raw):,} rows   "
      f"{raw.instance_date.min().date()} -> {raw.instance_date.max().date()}")
print(f"RAW residential units : {len(raw_res):,} rows")
print(f"CLEANED dataset       : {len(clean):,} rows   "
      f"{clean.instance_date.min().date()} -> {clean.instance_date.max().date()}")
print(f"\nCleaning removes {len(raw_res) - len(clean):,} rows "
      f"({(1 - len(clean)/len(raw_res))*100:.1f}%) — the reason counts use RAW.")

---
# 1. Transactions recorded each year
### (year-over-year transaction growth)

| | |
|---|---|
| **Dataset** | **RAW** — `transactions.parquet` |
| **Columns** | `instance_date`, `trans_group_en`, `property_type_en`, `property_usage_en` |
| **Filter** | `Sales` + `Unit` + `Residential`, years from 2011 |
| **Calculation** | count per year, then chained year-over-year growth |
| **Formula** | `((this year − previous year) ÷ previous year) × 100` |

2011 is the base year and carries no growth figure. The latest year is
incomplete and is handled separately in §1.3.

### 1.1 Filter and calculate → intermediate dataframe

In [ ]:
BASE_YEAR = 2011

# FILTER → count per (year, month) on the RAW registry
counts = (raw.assign(_res=RESIDENTIAL_UNIT_SALE)
             .groupby(["year", "month"], as_index=False)
             .agg(all_transactions=("_res", "size"),   # every registry transaction
                  transactions=("_res", "sum")))       # residential unit sales only
counts["year"]  = counts["year"].astype(int)
counts["month"] = counts["month"].astype(int)

# CALCULATION → one row per year
per_year = (counts.groupby("year", as_index=False)
                  .agg(transactions=("transactions", "sum"),
                       all_transactions=("all_transactions", "sum"),
                       months=("month", "nunique"))
                  .sort_values("year"))

latest_year   = int(per_year["year"].max())
latest_months = int(per_year.loc[per_year["year"] == latest_year, "months"].iloc[0])
per_year["complete"] = ~((per_year["year"] == latest_year) & (latest_months < 12))

# Chained growth: each year against the row directly above it.
per_year["yoy_pct"] = (per_year["transactions"] / per_year["transactions"].shift() - 1) * 100
# An incomplete year is not a comparable annual observation — drop its figure.
per_year.loc[~per_year["complete"], "yoy_pct"] = np.nan

yoy = per_year[per_year["year"] >= BASE_YEAR].reset_index(drop=True)
yoy.loc[0, "yoy_pct"] = np.nan          # base year has no predecessor

yoy          # INTERMEDIATE DATAFRAME

### 1.2 Result values

In [ ]:
print("Year   Transactions   All registry    YoY growth")
for _, r in yoy.iterrows():
    growth = "      base" if pd.isna(r.yoy_pct) else f"{r.yoy_pct:+9.2f}%"
    flag   = "   <- INCOMPLETE YEAR" if not r.complete else ""
    print(f"{int(r.year)}   {int(r.transactions):>12,}   {int(r.all_transactions):>12,}   {growth}{flag}")

complete = yoy[yoy.complete]
peak = complete.loc[complete.transactions.idxmax()]
best = complete.dropna(subset=["yoy_pct"]).nlargest(1, "yoy_pct").iloc[0]
print(f"\nBusiest completed year : {int(peak.year)} — {int(peak.transactions):,} transactions")
print(f"Strongest growth year  : {int(best.year)} — {best.yoy_pct:+.1f}% vs {int(best.year)-1}")

### 1.3 The incomplete-year rule

The latest year is compared with **the same months of the previous year** — the
only basis under which a part-year count can be compared with anything at all.

A percentage is released **only if that growth is strictly positive**. Zero or
negative is suppressed, because a shorter year is not a decline.

In [ ]:
cur_months = sorted(counts.loc[counts.year == latest_year, "month"].unique())
last_month = max(cur_months)
period     = f"January-{pd.Timestamp(2000, last_month, 1).strftime('%B')}"

current_n = int(counts.loc[counts.year == latest_year, "transactions"].sum())
basis_n   = int(counts.loc[(counts.year == latest_year - 1) &
                           (counts.month <= last_month), "transactions"].sum())
full_prev = int(counts.loc[counts.year == latest_year - 1, "transactions"].sum())

growth = (current_n / basis_n - 1) * 100
display_growth = growth if growth > 0 else None      # STRICTLY positive only

print(f"Latest year in the registry : {latest_year}")
print(f"Months available            : {len(cur_months)} of 12   ({period})")
print(f"Transactions so far         : {current_n:,}")
print(f"Same months of {latest_year-1}         : {basis_n:,}   <- the comparison basis")
print(f"{latest_year-1} full year              : {full_prev:,}")
print(f"Like-for-like growth        : {growth:+.2f}%")
print(f"\nStrictly positive?          : {growth > 0}")
print(f"Percentage displayed        : "
      f"{'YES, ' + format(display_growth, '+.1f') + '%' if display_growth else 'NO - count only'}")
print(f"Naive full-year comparison  : {(current_n/full_prev - 1)*100:+.1f}%"
      f"   <- never shown; a calendar artefact, not a market decline")

### 1.4 Visualisation

In [ ]:
done, todo = yoy[yoy.complete], yoy[~yoy.complete]

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=done.year, y=done.transactions, name="Transactions recorded",
                     marker_color="#2563EB",
                     text=[f"{v:,}" for v in done.transactions], textposition="outside",
                     hovertemplate="<b>%{x}</b><br>Transactions recorded: %{y:,}<extra></extra>"),
              secondary_y=False)
if not todo.empty:
    fig.add_trace(go.Bar(x=todo.year, y=todo.transactions,
                         name=f"{latest_year} - {period} only (in progress)",
                         marker_color="#D97706",
                         text=[f"{v:,}" for v in todo.transactions], textposition="outside",
                         hovertemplate=f"<b>%{{x}} - {period} only</b><br>"
                                       "Transactions so far: %{y:,}<br>"
                                       "<i>Part year - not comparable</i><extra></extra>"),
                  secondary_y=False)

line = yoy.dropna(subset=["yoy_pct"])[["year", "yoy_pct"]]
if display_growth is not None:
    line = pd.concat([line, pd.DataFrame([{"year": latest_year, "yoy_pct": display_growth}])])

fig.add_trace(go.Scatter(x=line.year, y=line.yoy_pct, name="Year-over-year growth (%)",
                         mode="lines+markers",
                         line=dict(color="#0D9488", width=2.2, dash="dot"),
                         hovertemplate="<b>%{x}</b><br>Growth vs previous year: "
                                       "%{y:+.1f}%<extra></extra>"),
              secondary_y=True)
fig.add_hline(y=0, line_color="rgba(128,128,128,0.45)", line_width=1, secondary_y=True)

fig.update_layout(height=470, barmode="group",
                  title="Transactions recorded each year (RAW registry)",
                  legend=dict(orientation="h", y=-0.24))
fig.update_yaxes(title_text="Transactions recorded", secondary_y=False)
fig.update_yaxes(title_text="Year-over-year growth (%)", secondary_y=True, showgrid=False)
fig.update_xaxes(title_text="Year", type="category")
fig.show()

**Interpretation.** Volume grew every year from 2021 to 2025. The contractions in
2014–2016, 2018 and 2020 are genuine and are kept. The latest year is drawn but
carries no growth figure, because it is not a full year.

---
# 2. How prices are moving

| | |
|---|---|
| **Dataset** | **CLEANED** — `latest_combined_data.parquet` |
| **Columns** | `year_month`, `actual_worth`, `meter_sale_price` |
| **Filter** | none beyond the dataset's own scope (the dashboard adds its sidebar filters) |
| **Calculation** | monthly medians, then a centred rolling median |
| **Formula** | `smoothed[t] = median(actual[t−6 … t+5])` for a 12-month centred window |

### 2.1 Monthly series → intermediate dataframe

In [ ]:
monthly = (clean.groupby("year_month", observed=True)
                .agg(transactions=("actual_worth", "size"),
                     median_price=("actual_worth", "median"),
                     median_rate=("meter_sale_price", "median"))
                .reset_index())
monthly = monthly[monthly["year_month"].astype(str).str.len() >= 6].copy()
monthly["period"] = pd.PeriodIndex(monthly["year_month"].astype(str), freq="M").to_timestamp()
monthly = monthly.sort_values("period").reset_index(drop=True)

SMOOTH_WINDOW = 12          # one full year — chosen by the test in the next cell
monthly["smooth_rate"]  = (monthly["median_rate"]
                           .rolling(SMOOTH_WINDOW, center=True, min_periods=1).median())
monthly["smooth_price"] = (monthly["median_price"]
                           .rolling(SMOOTH_WINDOW, center=True, min_periods=1).median())

monthly[["year_month", "transactions", "median_price", "smooth_price",
         "median_rate", "smooth_rate"]].tail(10)

### 2.2 Why a 12-month centred rolling **median** — the test that chose it

The window was measured, not assumed. Each candidate is scored on how much
month-to-month movement it removes and how far the resulting line sits from the
actual observations.

In [ ]:
s = monthly["median_rate"]
base_sd = s.pct_change().std() * 100

rows = []
for w in (3, 5, 7, 9, 12):
    for kind in ("median", "mean"):
        r  = s.rolling(w, center=True, min_periods=1)
        sm = r.median() if kind == "median" else r.mean()
        rows.append({
            "window": w, "statistic": kind,
            "sd of m/m change (%)": sm.pct_change().std() * 100,
            "calmer than actual (%)": (1 - sm.pct_change().std()*100 / base_sd) * 100,
            "median deviation from actual (%)": ((sm - s) / s * 100).abs().median(),
        })

print(f"Actual series: standard deviation of month-on-month change = {base_sd:.2f}%\n")
pd.DataFrame(rows)

**The choice, and why.**

- A **median**, not a mean: one unusual month should not bend the trend.
- A **12-month** window: it is one full year, so it removes the seasonal swing as
  well as the noise. It takes the month-on-month standard deviation from about
  7.6% to about 1.5% — roughly 80% calmer — while the trend still sits on the
  data (median deviation from the actual line ≈ 2.8%).
- `center=True` so the trend is not lagged; `min_periods=1` so the first and last
  months are still drawn, computed from a shorter window at the very ends.
- **Nothing is aggregated away.** Every monthly observation is retained, plotted
  and tabulated. The smoothed line is an extra series, not a replacement — this
  is explicitly *not* a conversion to multi-year intervals.

### 2.3 Result values

In [ ]:
actual_sd = monthly["median_rate"].pct_change().std() * 100
smooth_sd = monthly["smooth_rate"].pct_change().std() * 100
dev = ((monthly.smooth_rate - monthly.median_rate) / monthly.median_rate * 100).abs()

print(f"months in the series          : {len(monthly)}")
print(f"thinnest month                : {int(monthly.transactions.min()):,} transactions")
print(f"sd of m/m change - actual     : {actual_sd:.2f}%")
print(f"sd of m/m change - smoothed   : {smooth_sd:.2f}%   "
      f"({(1-smooth_sd/actual_sd)*100:.0f}% calmer)")
print(f"median |deviation| of trend   : {dev.median():.2f}%")
print(f"max |deviation| of trend      : {dev.max():.2f}%")
print("\nLast six months, actual vs 12-month trend:")
print(monthly[["year_month", "transactions", "median_rate", "smooth_rate"]]
      .tail(6).to_string(index=False))

### 2.4 Visualisation

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(x=monthly.period, y=monthly.median_price,
                         name="Median sale price - actual", mode="lines",
                         line=dict(color="#2563EB", width=1.0), opacity=0.35,
                         hovertemplate="%{x|%b %Y}<br>Actual median price: "
                                       "AED %{y:,.0f}<extra></extra>"),
              secondary_y=False)
fig.add_trace(go.Scatter(x=monthly.period, y=monthly.smooth_price,
                         name="Median sale price - 12-month trend", mode="lines",
                         line=dict(color="#2563EB", width=2.8),
                         hovertemplate="%{x|%b %Y}<br>Trend price: "
                                       "AED %{y:,.0f}<extra></extra>"),
              secondary_y=False)
fig.add_trace(go.Scatter(x=monthly.period, y=monthly.median_rate,
                         name="Median rate - actual", mode="lines",
                         line=dict(color="#0D9488", width=1.0), opacity=0.35,
                         hovertemplate="%{x|%b %Y}<br>Actual median rate: "
                                       "AED %{y:,.0f}/m2<extra></extra>"),
              secondary_y=True)
fig.add_trace(go.Scatter(x=monthly.period, y=monthly.smooth_rate,
                         name="Median rate - 12-month trend", mode="lines",
                         line=dict(color="#0D9488", width=2.8),
                         hovertemplate="%{x|%b %Y}<br>Trend rate: "
                                       "AED %{y:,.0f}/m2<extra></extra>"),
              secondary_y=True)

fig.update_layout(height=480, legend=dict(orientation="h", y=-0.24),
                  title="How prices are moving - actual monthly values and the 12-month trend")
fig.update_yaxes(title_text="Median sale price (AED)", secondary_y=False)
fig.update_yaxes(title_text="Median rate (AED/m2)", secondary_y=True, showgrid=False)
fig.show()

**Interpretation.** The faint lines are every actual month; the bold lines are the
12-month trend. Hovering the faint series still returns the original monthly
value, so the smoothing hides nothing.

---
# 3. Volume against price

| | |
|---|---|
| **Dataset** | **RAW** for volume · **CLEANED** for the rate |
| **Columns** | raw: the four count columns · clean: `year`, `meter_sale_price` |
| **Calculation** | count per year; **mean** rate per m² per year |
| **Formula** | `mean(meter_sale_price)` grouped by year |

**Why the mean here, when the rest of the analysis uses the median.** This chart
asks whether busy years are also expensive years — a question about the money
moving through the market as a whole. The mean reflects the entire distribution
including the upper tail that a hot market actually adds; the median deliberately
ignores it. Both are printed below so the difference stays visible.

### 3.1 Year | Transaction volume | Mean rate/m² → intermediate dataframe

In [ ]:
# Volume: RAW registry.  Rate: CLEANED dataset.  Each labelled at every step.
volume = (counts.groupby("year", as_index=False)
                .agg(transactions=("transactions", "sum"), months=("month", "nunique")))

rate = (clean.groupby("year", observed=True)["meter_sale_price"]
             .agg(mean_rate="mean", median_rate="median", priced_rows="size")
             .reset_index())
rate["year"] = rate["year"].astype(int)

vol_price = (volume.merge(rate, on="year", how="left")
                   .query("year >= @BASE_YEAR")
                   .sort_values("year").reset_index(drop=True))
vol_price["complete"] = ~((vol_price.year == latest_year) & (vol_price.months < 12))

vol_price[["year", "transactions", "mean_rate", "median_rate", "priced_rows", "complete"]]

### 3.2 Result values

In [ ]:
p = vol_price.dropna(subset=["mean_rate"])
print(f"correlation, volume vs MEAN rate   : {p.transactions.corr(p.mean_rate):.3f}")
print(f"correlation, volume vs MEDIAN rate : {p.transactions.corr(p.median_rate):.3f}")
print(f"mean above median in               : {(p.mean_rate > p.median_rate).sum()} of {len(p)} years")
print(f"average gap, mean - median         : AED {(p.mean_rate - p.median_rate).mean():,.0f}/m2")
print("\nYear | Transaction volume (RAW) | Mean rate/m2 (CLEANED)")
for _, r in vol_price.iterrows():
    tag = "" if r.complete else "  (part year)"
    print(f"{int(r.year)} | {int(r.transactions):>10,} | {r.mean_rate:>10,.0f}{tag}")

### 3.3 Visualisation

In [ ]:
done, todo = vol_price[vol_price.complete], vol_price[~vol_price.complete]

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=done.year, y=done.transactions, name="Transaction volume (raw)",
                     marker_color="#2563EB", opacity=0.85,
                     hovertemplate="<b>%{x}</b><br>Transaction volume: %{y:,}<extra></extra>"),
              secondary_y=False)
if not todo.empty:
    fig.add_trace(go.Bar(x=todo.year, y=todo.transactions,
                         name=f"{latest_year} - part year (in progress)",
                         marker_color="#D97706", opacity=0.9,
                         hovertemplate="<b>%{x} - part year</b><br>"
                                       "Volume so far: %{y:,}<extra></extra>"),
                  secondary_y=False)
fig.add_trace(go.Scatter(x=vol_price.year, y=vol_price.mean_rate,
                         name="Mean rate per m2 (cleaned)", mode="lines+markers",
                         line=dict(color="#0D9488", width=2.6), marker=dict(size=7),
                         hovertemplate="<b>%{x}</b><br>Mean rate: "
                                       "AED %{y:,.0f}/m2<extra></extra>"),
              secondary_y=True)

fig.update_layout(height=470, barmode="group", title="Volume against price, by year",
                  legend=dict(orientation="h", y=-0.24))
fig.update_yaxes(title_text="Transaction volume (raw registry)", secondary_y=False)
fig.update_yaxes(title_text="Mean rate per m2 (AED/m2)", secondary_y=True, showgrid=False)
fig.update_xaxes(title_text="Year", type="category")
fig.show()

**Interpretation.** Volume and the mean rate move together (correlation ≈ 0.88
across 16 years). That is an association across 16 observations, not evidence
that one causes the other.

---
# 4. Share of recorded transactions associated with each amenity
### (property type + amenity analysis)

| | |
|---|---|
| **Dataset** | **CLEANED** — the amenity flags exist only there |
| **Columns** | `rooms_en`, `has_parking`, `swimming_pool`, `balcony`, `elevator`, `metro` |
| **Filter 1** | property type — one at a time |
| **Filter 2** | amenity |
| **Formula** | `(rows where flag == 1) ÷ (rows of that property type) × 100` |

**This is not a purchase probability.** §4.1 is the check that establishes why,
and it must be run before the metric is interpreted.

### 4.1 Does the data contain a purchase / non-purchase outcome?

In [ ]:
import pyarrow.parquet as pq

raw_cols   = list(pq.ParquetFile(RAW_FILE).schema_arrow.names)
clean_cols = list(pq.ParquetFile(CLEAN_FILE).schema_arrow.names)

KEYWORDS = ("purchase", "buy", "bought", "outcome", "lead", "enquir", "inquir",
            "visit", "prospect", "convert", "target", "churn")
hits = sorted({c for c in raw_cols + clean_cols if any(k in c.lower() for k in KEYWORDS)})

print("Columns matching purchase / outcome keywords:", hits or "NONE")
print("\ntrans_group_en values in the RAW registry:")
print(pd.read_parquet(RAW_FILE, columns=["trans_group_en"])
        .trans_group_en.value_counts().to_string())
print("\nCONCLUSION")
print("  Every row in both files is a COMPLETED, RECORDED transaction.")
print("  There is no enquiry, viewing or non-purchase row anywhere.")
print("  => A customer purchase probability CANNOT be calculated from this data.")
print("  => The metric below is a SHARE OF RECORDED TRANSACTIONS, and nothing more.")

**What this means for the metric.** Transaction records describe deals that
completed. There is no row representing a buyer who looked and did not buy, so
there is no denominator from which a purchase probability could be estimated.
Anything calling itself a "customer purchase probability" here would be
fabricated. The share of recorded transactions is what the data actually
supports, and that is what is computed below.

### 4.2 Calculation → intermediate dataframe (within one property type)

In [ ]:
AMENITIES = {"has_parking": "Parking", "swimming_pool": "Swimming pool",
             "balcony": "Balcony", "elevator": "Elevator", "metro": "Near a metro station"}
PROPERTY_TYPE_LABELS = {"Studio": "Studio", "1 B/R": "1 BHK", "2 B/R": "2 BHK",
                        "3 B/R": "3 BHK", "4 B/R": "4 BHK", "5 B/R": "5 BHK",
                        "PENTHOUSE": "Penthouse"}
MIN_CELL = 100          # a group needs this many transactions before it is reported

def amenity_share(df, property_type):
    """Share of recorded transactions carrying each amenity flag, within ONE type."""
    sub = df[df.rooms_en == property_type]          # FILTER 1 - property type
    if len(sub) < MIN_CELL:
        return pd.DataFrame()
    rows = []
    for col, label in AMENITIES.items():            # FILTER 2 - amenity
        recorded = int((sub[col] == 1).sum())
        rows.append({"Amenity": label,
                     "Transactions with amenity recorded": recorded,
                     "Transactions without": len(sub) - recorded,
                     "Share of recorded transactions (%)": recorded / len(sub) * 100})
    return (pd.DataFrame(rows)
              .sort_values("Share of recorded transactions (%)", ascending=False)
              .reset_index(drop=True))

PROPERTY_TYPE = "1 B/R"          # <- change me
AMENITY_COL   = "has_parking"    # <- change me

within = amenity_share(clean, PROPERTY_TYPE)
within

### 4.3 The same amenity, across property types

In [ ]:
rows = []
for value, label in PROPERTY_TYPE_LABELS.items():
    sub = clean[clean.rooms_en == value]
    if len(sub) < MIN_CELL:
        continue
    recorded = int((sub[AMENITY_COL] == 1).sum())
    rows.append({"Property type": label, "Transactions": len(sub),
                 "With amenity recorded": recorded,
                 "Share of recorded transactions (%)": recorded / len(sub) * 100})
across = pd.DataFrame(rows)
across

### 4.4 Result values

In [ ]:
n = int((clean.rooms_en == PROPERTY_TYPE).sum())
print(f"{PROPERTY_TYPE_LABELS[PROPERTY_TYPE]} transactions on record: {n:,}\n")
for _, r in within.iterrows():
    print(f"  {r.Amenity:<22} {r['Share of recorded transactions (%)']:5.1f}%"
          f"   ({int(r['Transactions with amenity recorded']):,} of {n:,})")

print(f"\n{AMENITIES[AMENITY_COL]} across property types:")
for _, r in across.iterrows():
    print(f"  {r['Property type']:<12} {r['Share of recorded transactions (%)']:5.1f}%"
          f"   ({int(r['With amenity recorded']):,} of {int(r.Transactions):,})")

### 4.5 Visualisation

In [ ]:
t = within.iloc[::-1]
colors = ["#0D9488" if a == AMENITIES[AMENITY_COL] else "#94A3B8" for a in t.Amenity]

fig = go.Figure(go.Bar(
    x=t["Share of recorded transactions (%)"], y=t.Amenity, orientation="h",
    marker_color=colors,
    customdata=t[["Transactions with amenity recorded", "Transactions without"]],
    text=[f"{v:.1f}%" for v in t["Share of recorded transactions (%)"]],
    textposition="outside",
    hovertemplate="%{y}<br>Share of recorded transactions: %{x:.1f}%"
                  "<br>Recorded with: %{customdata[0]:,}"
                  "<br>Recorded without: %{customdata[1]:,}<extra></extra>"))
fig.update_layout(height=380,
                  title=f"Share of recorded {PROPERTY_TYPE_LABELS[PROPERTY_TYPE]} "
                        f"transactions associated with each amenity")
fig.update_xaxes(title_text="Share of recorded transactions (%)", range=[0, 112])
fig.show()

fig2 = go.Figure(go.Bar(
    x=across["Property type"], y=across["Share of recorded transactions (%)"],
    marker_color="#2563EB",
    text=[f"{v:.1f}%" for v in across["Share of recorded transactions (%)"]],
    textposition="outside",
    hovertemplate="%{x}<br>Share of recorded transactions: %{y:.1f}%<extra></extra>"))
fig2.update_layout(height=380, title=f"{AMENITIES[AMENITY_COL]} - share across property types")
fig2.update_yaxes(title_text=f"Share recorded with {AMENITIES[AMENITY_COL].lower()} (%)",
                  range=[0, 112])
fig2.update_xaxes(title_text="Property type")
fig2.show()

**Interpretation.** Read each bar as "this percentage of recorded sales of that
type had the flag on the record". It describes what completed records contain.
It is not a probability of purchase, it says nothing about price, and a `0` means
*not recorded* — which is not the same as *confirmed absent*, so under-recording
appears as a lower share.

---
# 5. Rate by building height and property type

| | |
|---|---|
| **Dataset** | **CLEANED** |
| **Columns** | `floors`, `building_name_en`, `rooms_en`, `meter_sale_price` |
| **Filter** | rows with a height value; cells with ≥100 transactions |
| **Buckets** | quartiles of the height distribution, one row per building |

§5.1 establishes what the field actually is before anything is bucketed.

### 5.1 Is the field the unit's floor, or the building's height?

In [ ]:
floor_check = pd.read_parquet(CLEAN_FILE, columns=["floor_bin", "floors", "property_id_bld"])

print("floor_bin distinct values :", floor_check.floor_bin.dropna().unique().tolist())
print(f"floor_bin populated       : {floor_check.floor_bin.notna().mean()*100:.1f}% of rows,")
print("                            and every populated row is the string 'Unknown'")

per_bld = (floor_check.dropna(subset=["floors", "property_id_bld"])
                      .groupby("property_id_bld")["floors"].nunique())
print(f"\nBuildings with a height value      : {len(per_bld):,}")
print(f"Height CONSTANT within a building  : {(per_bld == 1).mean()*100:.1f}%")
print(f"floors range                       : {floor_check.floors.min():.0f} - "
      f"{floor_check.floors.max():.0f}")
print(f"floors populated                   : {floor_check.floors.notna().mean()*100:.1f}% of rows")

print("\nCONCLUSION")
print("  `floors` is identical for every sale in a given building, so it is the")
print("  BUILDING'S HEIGHT, not the unit's own floor. `floor_bin` carries no")
print("  information at all. A floor-level analysis is NOT possible with this data.")
print("  The analysis below is building height, and it is labelled that way.")

### 5.2 Data-driven bucket boundaries

In [ ]:
hgt = clean.dropna(subset=["floors"]).copy()

# Bucket boundaries from the DATA: quartiles of the height distribution taken
# ONE ROW PER BUILDING, so a tower with thousands of sales cannot move them.
per_building = hgt.groupby("building_name_en", observed=True)["floors"].first()
q1, q2, q3 = (int(round(per_building.quantile(q))) for q in (0.25, 0.50, 0.75))
edges  = [0, q1, q2, q3, int(per_building.max())]
labels = [f"Low-rise (<={q1} floors)", f"Mid-rise ({q1+1}-{q2})",
          f"High-rise ({q2+1}-{q3})", f"Tower (>{q3} floors)"]

print(f"buildings with a height : {len(per_building):,}")
print(f"per-building quartiles  : p25={q1}  p50={q2}  p75={q3}  max={int(per_building.max())}")
print(f"bucket edges            : {edges}")

hgt["height_band"] = pd.cut(hgt.floors, bins=edges, labels=labels,
                            include_lowest=True, right=True)
hgt = hgt[hgt.rooms_en.isin(PROPERTY_TYPE_LABELS)]

print("\ntransactions per band:")
print(hgt.height_band.value_counts().reindex(labels).to_string())

### 5.3 Calculation → intermediate dataframe

In [ ]:
height_table = (hgt.groupby(["height_band", "rooms_en"], observed=True)
                   .agg(median_rate=("meter_sale_price", "median"),
                        mean_rate=("meter_sale_price", "mean"),
                        transactions=("meter_sale_price", "size"))
                   .reset_index())

dropped      = height_table[height_table.transactions <  MIN_CELL]
height_table = height_table[height_table.transactions >= MIN_CELL].copy()
height_table["Property type"] = height_table.rooms_en.map(PROPERTY_TYPE_LABELS)

print(f"height recorded for {len(hgt):,} of {len(clean):,} transactions "
      f"({len(hgt)/len(clean)*100:.1f}%)")
print(f"\ncells below the {MIN_CELL}-transaction threshold — omitted and named:")
for _, r in dropped[dropped.transactions > 0].iterrows():
    print(f"   {PROPERTY_TYPE_LABELS.get(r.rooms_en, r.rooms_en)} in {r.height_band} "
          f"({int(r.transactions)} transactions)")

height_table.pivot(index="height_band", columns="Property type", values="median_rate").round(0)

### 5.4 Visualisation

In [ ]:
palette = ["#2563EB", "#0D9488", "#D97706", "#7C3AED", "#DC2626", "#059669", "#DB2777"]
order   = [l for l in PROPERTY_TYPE_LABELS.values() if l in set(height_table["Property type"])]

fig = go.Figure()
for i, ptype in enumerate(order):
    d = height_table[height_table["Property type"] == ptype]
    fig.add_trace(go.Bar(x=d.height_band.astype(str), y=d.median_rate, name=ptype,
                         marker_color=palette[i % len(palette)],
                         customdata=d[["transactions", "mean_rate"]],
                         hovertemplate=f"<b>{ptype}</b><br>%{{x}}<br>"
                                       "Median rate: AED %{y:,.0f}/m2<br>"
                                       "Mean rate: AED %{customdata[1]:,.0f}/m2<br>"
                                       "Transactions: %{customdata[0]:,}<extra></extra>"))

fig.update_layout(height=490, barmode="group",
                  title="Rate by building height and property type",
                  legend=dict(orientation="h", y=-0.26))
fig.update_yaxes(title_text="Median rate (AED/m2)")
fig.update_xaxes(title_text="Building height band")
fig.show()

**Interpretation.** The rate rises with building height for every property type.
Two caveats travel with that: height and location are entangled — tall towers
cluster in particular areas — and this cannot answer whether a higher floor beats
a lower one inside the same tower, because the unit's floor is not recorded.

---
# 6. Where the price points are

| | |
|---|---|
| **Dataset** | **CLEANED** |
| **Columns** | `actual_worth` |
| **Bands** | seven fixed bands, **left-closed / right-open** |

Left-closed means a sale of exactly AED 1,000,000 is counted in `1M – 2M`, never
in `500K – 1M`. The audit proves the bands are exhaustive and mutually
exclusive.

### 6.1 Calculation → intermediate dataframe

In [ ]:
BAND_EDGES  = [0, 500_000, 1_000_000, 2_000_000, 3_000_000, 5_000_000, 10_000_000, np.inf]
BAND_LABELS = ["< 500K", "500K - 1M", "1M - 2M", "2M - 3M", "3M - 5M", "5M - 10M", "> 10M"]

# right=False makes every band LEFT-CLOSED / RIGHT-OPEN: a sale of exactly
# AED 1,000,000 lands in "1M - 2M", never in "500K - 1M".
band  = pd.cut(clean.actual_worth, bins=BAND_EDGES, labels=BAND_LABELS, right=False)
bands = (band.value_counts().reindex(BAND_LABELS).fillna(0).astype(int)
             .rename_axis("Price band (AED)").reset_index(name="Transactions"))
bands["Share (%)"] = bands.Transactions / len(clean) * 100
bands

### 6.2 Result values — the audit, and what the range means

In [ ]:
p = clean.actual_worth.dropna()

print("AUDIT - the bands must account for every row exactly once")
print(f"  rows in the dataset       : {len(clean):,}")
print(f"  rows assigned to a band   : {int(bands.Transactions.sum()):,}")
print(f"  unassigned                : {len(clean) - int(bands.Transactions.sum()):,}")
print(f"  shares sum to             : {bands['Share (%)'].sum():.2f}%")

print("\nTHE RANGE, in the terms the dashboard uses")
print(f"  observations              : {len(p):,}")
print(f"  minimum (cheapest sale)   : AED {p.min():,.0f}")
print(f"  maximum (dearest sale)    : AED {p.max():,.0f}")
print(f"  25th percentile           : AED {p.quantile(.25):,.0f}")
print(f"  median                    : AED {p.median():,.0f}")
print(f"  75th percentile           : AED {p.quantile(.75):,.0f}")
print(f"  middle half of the market : AED {p.quantile(.25):,.0f} - {p.quantile(.75):,.0f}")

lo, hi = p.quantile([0.005, 0.995])
print(f"\nHISTOGRAM DISPLAY RANGE (the companion chart)")
print(f"  0.5th - 99.5th percentile : AED {lo:,.0f} - {hi:,.0f}")
print(f"  bins                      : 60 equal-width")
print(f"  bin width                 : AED {(hi-lo)/60:,.0f}")
print(f"  rows outside the display  : {int(((p < lo) | (p > hi)).sum()):,} "
      f"({((p < lo) | (p > hi)).mean()*100:.1f}%) - still in every statistic above")

### 6.3 Visualisation

In [ ]:
fig = go.Figure(go.Bar(x=bands["Price band (AED)"], y=bands.Transactions,
                       marker_color="#2563EB",
                       text=[f"{v:.1f}%" for v in bands["Share (%)"]], textposition="outside",
                       hovertemplate="%{x}<br>Transactions: %{y:,}<extra></extra>"))
fig.update_layout(height=410, title="Where the price points are")
fig.update_yaxes(title_text="Transactions")
fig.update_xaxes(title_text="Price band (AED)")
fig.show()

**Interpretation of the range.** The minimum and the maximum are each a single
real transaction, not a typical one. The middle half of the market sits between
the 25th and 75th percentiles, and the median is the typical transaction. On the
companion histogram the display span is the 0.5th–99.5th percentile cut into 60
equal bins — the 1% outside that span is still counted in every statistic above,
it is simply off the ends of the picture.

---
# 7. Rate per m² by layout

| | |
|---|---|
| **Dataset** | **CLEANED** |
| **Columns** | `rooms_en`, `meter_sale_price`, `procedure_area` |
| **Filter** | layouts with ≥100 transactions; smaller ones are listed, not deleted |
| **Calculation** | 25th percentile, median, 75th percentile, whiskers at 1.5 × IQR |

**Why the median and quartiles rather than the mean.** Rate per m² is
right-skewed — a handful of very expensive units pull an average upward. The
median is the typical transaction, and the quartiles show how wide each layout's
market actually is, which a single number cannot.

### 7.1 Grouping and aggregation → intermediate dataframe

In [ ]:
ROOM_ORDER = ["Studio", "1 B/R", "2 B/R", "3 B/R", "4 B/R", "5 B/R",
              "6 B/R", "7 B/R", "PENTHOUSE"]
MIN_LAYOUT = 100        # layouts below this are listed, never silently dropped

lay = clean.dropna(subset=["rooms_en", "meter_sale_price"])
counts_by_layout = lay.rooms_en.value_counts()
kept    = [r for r in ROOM_ORDER if counts_by_layout.get(r, 0) >= MIN_LAYOUT]
excluded = [r for r in ROOM_ORDER if 0 < counts_by_layout.get(r, 0) < MIN_LAYOUT]

g = lay[lay.rooms_en.isin(kept)].groupby("rooms_en", observed=True)["meter_sale_price"]
stats = g.quantile([0.25, 0.50, 0.75]).unstack()
stats.columns = ["q1", "median", "q3"]
stats["iqr"] = stats.q3 - stats.q1

# Whiskers reach the furthest ACTUAL observation inside 1.5 x IQR - no value is
# deleted, the whisker simply stops at the last ordinary transaction.
lo_fence = (stats.q1 - 1.5 * stats.iqr).clip(lower=float(lay.meter_sale_price.min()))
hi_fence =  stats.q3 + 1.5 * stats.iqr
stats["lower_whisker"] = [float(g.get_group(k)[g.get_group(k) >= lo_fence[k]].min())
                          for k in stats.index]
stats["upper_whisker"] = [float(g.get_group(k)[g.get_group(k) <= hi_fence[k]].max())
                          for k in stats.index]
stats["transactions"]  = g.size()
stats["mean"]          = g.mean()
stats["median_size_m2"] = (lay[lay.rooms_en.isin(kept)]
                           .groupby("rooms_en", observed=True)["procedure_area"].median())
stats = stats.reindex(kept)
stats.round(0)

### 7.2 Result values

In [ ]:
print("Layout      Transactions   25th pct     Median       75th pct    Median size")
for name, r in stats.iterrows():
    print(f"{str(name):<11} {int(r.transactions):>11,}   {r.q1:>9,.0f}   "
          f"{r['median']:>9,.0f}   {r.q3:>9,.0f}   {r.median_size_m2:>7,.0f} m2")

if excluded:
    print(f"\nNot plotted - fewer than {MIN_LAYOUT} transactions "
          f"(listed, not deleted):")
    for r in excluded:
        sub = lay.loc[lay.rooms_en == r, "meter_sale_price"]
        print(f"  {r}: {len(sub)} transactions, median AED {sub.median():,.0f}/m2")

print(f"\nDoes the rate rise with layout size?")
ladder = [r for r in ["Studio", "1 B/R", "2 B/R", "3 B/R", "4 B/R"] if r in stats.index]
vals = [round(float(stats.loc[r, "median"]), 1) for r in ladder]
print(f"  {dict(zip(ladder, vals))}")
print(f"  monotonically increasing: {vals == sorted(vals)}")

### 7.3 Visualisation

In [ ]:
palette = ["#2563EB", "#0D9488", "#D97706", "#7C3AED", "#DC2626", "#059669", "#DB2777"]

fig = go.Figure()
for i, name in enumerate(stats.index):
    r = stats.loc[name]
    fig.add_trace(go.Box(
        x=[str(name)], q1=[r.q1], median=[r["median"]], q3=[r.q3],
        lowerfence=[r.lower_whisker], upperfence=[r.upper_whisker],
        name=f"{name} ({int(r.transactions):,})",
        marker_color=palette[i % len(palette)], width=0.55,
        hovertemplate=(f"<b>{name}</b><br>Upper whisker: %{{upperfence:,.0f}}"
                       "<br>75th pct: %{q3:,.0f}<br>Median: %{median:,.0f}"
                       "<br>25th pct: %{q1:,.0f}"
                       "<br>Lower whisker: %{lowerfence:,.0f}<extra></extra>")))

fig.update_layout(height=470, title="Rate per m2 by layout",
                  legend=dict(orientation="h", y=-0.22))
fig.update_yaxes(title_text="Rate per m2 (AED)")
fig.update_xaxes(title_text="Layout")
fig.show()

**Interpretation.** The median rate rises with layout size — larger apartments in
Dubai are dearer per square metre, not cheaper. The boxes overlap heavily, so
layout alone explains only part of the variation in rate.

---
# 8. Unit size — key statistics

| | |
|---|---|
| **Dataset** | **CLEANED** |
| **Columns** | `rooms_en`, `procedure_area` |
| **Calculation** | count, min, 25th pct, median, mean, 75th pct, max per property type |

### 8.1 Calculation → intermediate dataframe

In [ ]:
order = [v for v in PROPERTY_TYPE_LABELS if v in set(clean.rooms_en.dropna().unique())]

size_stats = (clean[clean.rooms_en.isin(order)]
              .groupby("rooms_en", observed=True)["procedure_area"]
              .agg(Transactions="size",
                   Smallest="min",
                   p25=lambda s: s.quantile(0.25),
                   Median="median",
                   Mean="mean",
                   p75=lambda s: s.quantile(0.75),
                   Largest="max")
              .reindex(order).dropna(how="all").reset_index())
size_stats["Property type"] = size_stats.rooms_en.map(PROPERTY_TYPE_LABELS)
size_stats = size_stats[["Property type", "Transactions", "Smallest", "p25",
                         "Median", "Mean", "p75", "Largest"]]
size_stats.round(1)

### 8.2 Result values — the whole selection

In [ ]:
overall = clean.procedure_area.dropna()
print(f"observations              : {len(overall):,}")
print(f"minimum                   : {overall.min():,.0f} m2")
print(f"25th percentile           : {overall.quantile(.25):,.0f} m2")
print(f"median                    : {overall.median():,.0f} m2")
print(f"mean                      : {overall.mean():,.1f} m2")
print(f"75th percentile           : {overall.quantile(.75):,.0f} m2")
print(f"maximum                   : {overall.max():,.0f} m2")
print(f"middle half of the market : {overall.quantile(.25):,.0f} - "
      f"{overall.quantile(.75):,.0f} m2")
print(f"mean above median by      : {overall.mean() - overall.median():,.1f} m2 "
      f"- the right tail of large units")

**Interpretation.** The median is the typical size for each property type; the
25th–75th percentile columns bracket the middle half. The smallest and largest
columns are single transactions and are not typical. The mean sits above the
median because a small number of very large units pull it up.

---
# 9. Sale price by registration type — summary

| | |
|---|---|
| **Dataset** | **CLEANED** |
| **Columns** | `reg_type_en`, `actual_worth`, `meter_sale_price` |
| **Calculation** | count, share, 25th pct, median, 75th pct of price; median rate |

Each row is described on its own terms. **No difference between the two rows is
computed, and no premium or discount is stated** — the two segments are different
products in different buildings, so a difference between the rows would not be a
like-for-like comparison.

### 9.1 Calculation → intermediate dataframe

In [ ]:
reg_summary = (clean.groupby("reg_type_en", observed=True)
                    .agg(Transactions=("actual_worth", "size"),
                         p25=("actual_worth", lambda s: s.quantile(0.25)),
                         Median=("actual_worth", "median"),
                         p75=("actual_worth", lambda s: s.quantile(0.75)),
                         MedianRate=("meter_sale_price", "median"))
                    .reset_index()
                    .rename(columns={"reg_type_en": "Registration type"}))
reg_summary["Share of transactions (%)"] = (reg_summary.Transactions
                                            / reg_summary.Transactions.sum() * 100)
reg_summary = reg_summary[["Registration type", "Transactions",
                           "Share of transactions (%)", "p25", "Median", "p75",
                           "MedianRate"]]
reg_summary.columns = ["Registration type", "Transactions", "Share of transactions (%)",
                       "25th pct price (AED)", "Median price (AED)",
                       "75th pct price (AED)", "Median rate (AED/m2)"]
reg_summary.round(1)

### 9.2 Result values

In [ ]:
for _, r in reg_summary.iterrows():
    print(f"{r['Registration type']}")
    print(f"   transactions          : {int(r.Transactions):,} "
          f"({r['Share of transactions (%)']:.1f}% of the selection)")
    print(f"   middle half of prices : AED {r['25th pct price (AED)']:,.0f} - "
          f"{r['75th pct price (AED)']:,.0f}")
    print(f"   median price          : AED {r['Median price (AED)']:,.0f}")
    print(f"   median rate           : AED {r['Median rate (AED/m2)']:,.0f}/m2")
    print()
print("Each row is described on its own terms. No difference between the two rows")
print("is computed here, and no premium or discount is stated.")

---
# 10. How the price distribution has changed

| | |
|---|---|
| **Dataset** | **CLEANED** |
| **Columns** | `year`, `meter_sale_price` |
| **Method** | deterministic 45,000-row sample, one violin per year |
| **Outlier handling** | none — no value is trimmed or removed |

The dashboard samples for responsiveness. §10.1 measures whether the sample is
faithful; that measurement is what decided the methodology was sound and should
be preserved rather than replaced.

### 10.1 Is the sample faithful to the population?

In [ ]:
SAMPLE_N = 45_000
sample = clean.sample(SAMPLE_N, random_state=42)     # deterministic

fidelity = pd.DataFrame({
    "population n":      clean.groupby("year").size(),
    "sample n":          sample.groupby("year").size(),
    "population median": clean.groupby("year")["meter_sale_price"].median(),
    "sample median":     sample.groupby("year")["meter_sale_price"].median(),
    "population p75":    clean.groupby("year")["meter_sale_price"].quantile(0.75),
    "sample p75":        sample.groupby("year")["meter_sale_price"].quantile(0.75),
}).dropna()
fidelity["median error (%)"] = (fidelity["sample median"] / fidelity["population median"] - 1) * 100
fidelity["p75 error (%)"]    = (fidelity["sample p75"]    / fidelity["population p75"]    - 1) * 100
fidelity.round(2)

### 10.2 Result values

In [ ]:
print(f"sample size                 : {SAMPLE_N:,} of {len(clean):,} rows "
      f"({SAMPLE_N/len(clean)*100:.1f}%), random_state=42")
print(f"worst per-year median error : {fidelity['median error (%)'].abs().max():.2f}%")
print(f"worst per-year p75 error    : {fidelity['p75 error (%)'].abs().max():.2f}%")
print(f"smallest per-year sample    : {int(fidelity['sample n'].min()):,} rows")

print("\nPer-year distribution shape on the FULL population (no sampling):")
shape = (clean.groupby("year")["meter_sale_price"]
              .agg(n="size", p25=lambda s: s.quantile(.25), median="median",
                   p75=lambda s: s.quantile(.75)))
shape["iqr"] = shape.p75 - shape.p25
shape["iqr_as_pct_of_median"] = shape.iqr / shape["median"] * 100
print(shape.round(0).to_string())
print("\n=> The sample tracks the population closely, so the existing sampled")
print("   violin methodology is sound and was preserved rather than replaced.")

### 10.3 Visualisation

In [ ]:
fig = go.Figure()
for y in sorted(sample.year.unique()):
    fig.add_trace(go.Violin(y=sample.loc[sample.year == y, "meter_sale_price"],
                            name=str(int(y)), box_visible=True, points=False,
                            meanline_visible=False,
                            hovertemplate="Year %{x}<br>Rate: AED %{y:,.0f}/m2<extra></extra>"))
fig.update_layout(height=490, showlegend=False,
                  title="How the price distribution has changed (rate per m2, by year)")
fig.update_yaxes(title_text="Rate per m2 (AED/m2)")
fig.update_xaxes(title_text="Year")
fig.show()

**Interpretation.** Each shape is one year. The width at a given height is how
many transactions sat at that rate. Two things are visible at once: the centre
moving upward, and the shape widening — the market has not only risen, it has
spread out. A median line alone would show only the first.

---
# 11. Year-by-year summary

| | |
|---|---|
| **Dataset** | **RAW** for the count · **CLEANED** for the rates |
| **Columns** | raw: the four count columns · clean: `year`, `meter_sale_price` |
| **Calculation** | `count`, `mean`, `median` per year |

**Why two sources in one table.** A transaction count must state how many
transactions were recorded, and preprocessing removes rows from the cleaned file
— so the count comes from the raw registry. Rate statistics need the validated
price basis and the engineered columns, so they come from the cleaned dataset.
Each column is labelled, and the number of priced rows actually behind the rate
columns is shown so the two can never be confused.

### 11.1 Calculation → the table

In [ ]:
summary = vol_price[["year", "transactions", "mean_rate", "median_rate",
                     "priced_rows", "complete"]].copy()
summary["Year"] = summary["year"].astype(int).astype(str)
summary.loc[~summary.complete, "Year"] += f"  ({period}, in progress)"
summary = summary[["Year", "transactions", "mean_rate", "median_rate", "priced_rows"]]
summary.columns = ["Year", "Number of transactions (raw)", "Mean rate/m2 (AED)",
                   "Median rate/m2 (AED)", "Priced transactions used (cleaned)"]
summary

### 11.2 Result values

In [ ]:
c = vol_price[vol_price.complete].dropna(subset=["median_rate"])
hi, lo = c.loc[c.median_rate.idxmax()], c.loc[c.median_rate.idxmin()]
latest_row = vol_price.dropna(subset=["median_rate"]).iloc[-1]

print(f"Highest median rate : AED {hi.median_rate:,.0f}/m2 in {int(hi.year)}")
print(f"Lowest median rate  : AED {lo.median_rate:,.0f}/m2 in {int(lo.year)}")
print(f"Latest ({int(latest_row.year)})       : AED {latest_row.median_rate:,.0f}/m2 "
      f"({(latest_row.median_rate/lo.median_rate - 1)*100:+.0f}% vs the lowest year)")

gap = vol_price.dropna(subset=["mean_rate"])
print(f"\nMean above median in {(gap.mean_rate > gap.median_rate).sum()} of {len(gap)} years")
print(f"RAW count exceeds the priced CLEANED rows in "
      f"{(vol_price.transactions > vol_price.priced_rows).sum()} of {len(vol_price)} years")
print("  -> the two columns come from different files, and that is why they differ")

**Interpretation.** The mean sits above the median in every year — the effect of a
small number of very large deals. The median is the figure to quote for a typical
transaction. The final row is the year in progress and is labelled with the
period it covers.

---
# Appendix — cross-check against the dashboard

Not a visualisation section. This restates the headline figures the notebook
produced so they can be read against the dashboard side by side, and states where
a legitimate difference is expected.

In [ ]:
print("TRANSACTION COUNTS (RAW registry - should match the dashboard exactly)")
for y in (2011, 2015, 2020, 2024, 2025, latest_year):
    row = yoy[yoy.year == y]
    if not row.empty:
        print(f"   {y}: {int(row.transactions.iloc[0]):,}")

print(f"\nINCOMPLETE-YEAR RULE")
print(f"   {latest_year} period          : {period}  ({len(cur_months)} of 12 months)")
print(f"   transactions so far   : {current_n:,}")
print(f"   like-for-like growth  : {growth:+.2f}%")
print(f"   percentage displayed  : {'YES' if display_growth else 'NO - count only'}")

print("\nPRICE FIGURES (CLEANED, UNFILTERED)")
print(f"   median rate           : AED {clean.meter_sale_price.median():,.0f}/m2")
print(f"   mean rate             : AED {clean.meter_sale_price.mean():,.0f}/m2")
print(f"   median sale price     : AED {clean.actual_worth.median():,.0f}")
print(f"   rows                  : {len(clean):,}")

print("\nNOTE ON DIFFERENCES")
print("   Transaction counts come from the raw registry and are not filtered in")
print("   either place, so they should match the dashboard exactly.")
print("   Price figures here are computed on the UNFILTERED cleaned dataset. The")
print("   dashboard applies its sidebar filters, and the sale-price and unit-size")
print("   sliders default to the 1st-99th percentile - so its price figures will")
print("   differ slightly until those sliders are widened to the full range.")